# 01. Ingesta de datos
**Objetivo:** descargar los CSV desde https://mibici.net/es/datos-abiertos con un rango de fecha establecido, identificar diferentes esquemas y codificación entre los diferentes CSVs.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import requests

from mi_bici import data as ds
from mi_bici.config import RAW_DATA_DIR

pd.set_option('display.max_colwidth', None)


2026-10-16 21:56:27.223 | INFO     | mi_bici.config:<module>:11 - PROJ_ROOT path is: /home/chuy/Documents/mna/mlops/TC5061-Proyecto-Equipo11


## 1. Descargar datos

In [2]:
# Descargar un archivo de prueba

url = 'https://mibici.net/site/assets/files/61450/datos_abiertos_2024_01.csv'

try:
    with requests.get(url, stream=True, timeout=30) as response:
        response.raise_for_status()
except requests.HTTPError as error:
    print(f'{type(error).__name__}: {error}')

HTTPError: 406 Client Error: Not Acceptable for url: https://mibici.net/site/assets/files/61450/datos_abiertos_2024_01.csv


In [3]:
# ¿Porqué no se pueden descargar los archivos?

user_agents = ['python-requests/2.32', ds.HTTP_HEADERS['User-Agent'], 'Mozilla/5.0 (X11; Linux x86_64)']

rows = []
for user_agent in user_agents:
    with requests.get(url, headers={'User-Agent': user_agent}, stream=True, timeout=30) as response:
        rows.append({'user_agent': user_agent, 'status_code': response.status_code})

pd.DataFrame(rows)

,user_agent,status_code
0,python-requests/2.32,406
1,mi-bici-tc5061-equipo11,200
2,Mozilla/5.0 (X11; Linux x86_64),200


In [4]:
# Definir rango
START = '2024-01'
END = '2026-09'

sources = ds.load_sources()
trip_urls = ds.select_trip_urls(sources, START, END)

print(f'Meses a descargar: {len(trip_urls)}')

Meses a descargar: 33


In [5]:
# Descargar archivos
stations_path = ds.download_stations(sources)
trip_paths = ds.download_trips(trip_urls)

print(f'Catálogo de estaciones: {stations_path.name}')
print(f'Meses descargados: {len(trip_paths)}')

Catálogo de estaciones: nomenclatura_2026_09.csv
Meses descargados: 33


In [6]:
# Revisar peso de los archivos

files = pd.DataFrame(
    [
        {'year_month': year_month, 'file': path.name, 'MB': path.stat().st_size / 1e6}
        for year_month, path in trip_paths.items()
    ]
)
sizes = files['MB']

print(f'Archivos por mes - Total: {files["MB"].sum():,.1f} MB')
print(f'Archivos por mes - Promedio: {files["MB"].mean():,.1f} MB, mínimo {files["MB"].min():,.1f}, máximo {files["MB"].max():,.1f}')
print(f'Catálogo de estaciones: {stations_path.stat().st_size / 1e3:,.1f} KB')

Archivos por mes - Total: 1,070.1 MB
Archivos por mes - Promedio: 32.4 MB, mínimo 25.4, máximo 42.6
Catálogo de estaciones: 40.8 KB


## 2. Viajes

In [7]:
# Ver muestra

ds.read_raw_csv(trip_paths[START], nrows=5, dtype=str)

,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
0,31789291,640715,M,1984,2024-01-01 00:05:25,2024-01-01 00:29:37,294,271
1,31789292,2353010,F,1995,2024-01-01 00:07:23,2024-01-01 00:16:37,182,254
2,31789293,1556365,M,1984,2024-01-01 00:07:31,2024-01-01 00:16:37,182,254
3,31789294,2626233,M,1994,2024-01-01 00:07:56,2024-01-01 00:31:51,35,154
4,31789295,2602006,M,1998,2024-01-01 00:08:09,2024-01-01 00:31:51,35,154


In [8]:
# Revisar encoding

header_summary = ds.compare_headers(trip_paths)
print(header_summary['encoding'].value_counts().to_string())

encoding
utf-8-sig    18
cp1252       15


In [9]:
# Identificar diferencias en los encabezados de los archivos

print('Número de columnas por archivo:')
print(header_summary['n_columns'].value_counts().to_string())

header_variants = header_summary.groupby('header_variant').agg(
    n_files=('year_month', 'count'),
    year_months=('year_month', list),
    columns=('columns', 'first'),
)
header_variants

Número de columnas por archivo:
n_columns
8    33


,n_files,year_months,columns
header_variant,,,
0,31,"[2024-01, 2024-03, 2024-04, 2024-06, 2024-07, 2024-08, 2024-09, 2024-10, 2024-11, 2024-12, 2025-01, 2025-02, 2025-03, 2025-04, 2025-05, 2025-06, 2025-07, 2025-08, 2025-09, 2025-10, 2025-11, 2025-12, 2026-01, 2026-02, 2026-03, 2026-04, 2026-05, 2026-06, 2026-07, 2026-08, 2026-09]","(Viaje_Id, Usuario_Id, Genero, Año_de_nacimiento, Inicio_del_viaje, Fin_del_viaje, Origen_Id, Destino_Id)"
1,2,"[2024-02, 2024-05]","(Viaje_Id, Usuario_Id, Genero, A}äe_nacimiento, Inicio_del_viaje, Fin_del_viaje, Origen_Id, Destino_Id)"


In [10]:
# Identificar formatos de fecha

# Esta tabla nos ayudará para comparar un solo valor por archivo, en el mejor de los casos
# Como son millones de registros, comparar fecha por fecha va a consumir tiempo y recursos
DIGITS_TO_NINE = str.maketrans('0123456789', '9' * 10)

date_shapes = []
for year_month, path in trip_paths.items():
    raw = pd.read_csv(path, encoding=ds.detect_encoding(path), usecols=[4, 5], header=0, dtype=str)
    values = pd.concat([raw.iloc[:, 0], raw.iloc[:, 1]]).dropna()
    shapes = values.str.translate(DIGITS_TO_NINE)
    for shape, n_values in shapes.value_counts().items():
        shape_values = values[shapes == shape]

        first_part_len = len(shape) - len(shape.lstrip('9'))
        date_shapes.append(
            {
                'year_month': year_month,
                'shape': shape,
                'n_values': n_values,
                'max_part_1': shape_values.str.slice(0, first_part_len).astype(int).max(),
                'example': shape_values.iloc[0],
            }
        )

date_shapes = pd.DataFrame(date_shapes)
date_shapes.groupby('shape').agg(
    n_files=('year_month', 'nunique'),
    n_values=('n_values', 'sum'),
    max_part_1=('max_part_1', 'max'),
    example=('example', 'first'),
    year_months=('year_month', lambda s: ', '.join(s.unique())),
)

,n_files,n_values,max_part_1,example,year_months
shape,,,,,
99/99/9999 99:99,2,1245609,31,01/09/2024 10:00,"2024-09, 2024-10"
99/99/9999 9:99,2,371915,31,01/09/2024 0:00,"2024-09, 2024-10"
9999-99-99 99:99:99,31,23462512,2026,2024-01-01 00:05:25,"2024-01, 2024-02, 2024-03, 2024-04, 2024-05, 2024-06, 2024-07, 2024-08, 2024-11, 2024-12, 2025-01, 2025-02, 2025-03, 2025-04, 2025-05, 2025-06, 2025-07, 2025-08, 2025-09, 2025-10, 2025-11, 2025-12, 2026-01, 2026-02, 2026-03, 2026-04, 2026-05, 2026-06, 2026-07, 2026-08, 2026-09"


## Hallazgos
- El servidor de MiBico bloquea los requests on el user agent default de `request`.
- El peso promedio de los archivos de cada año_mes es de 32.4 MB.
- Se identifican 2 tipos de encodings: utf-8-sig (18 archivos) y cp1252 (15 archivos).
- Se identifican 2 variantes de encabezados. La diferencia es en el nombre de la columna "Año_de_nacimiento". Hay dos archivos con "A}äe_nacimiento".
- El número de columnas (8) y el orden de las columnas se mantiene en los 33 archivos descargados.
- Se identifican 2 formatos diferente de fecha. AAAA-MM-DD 00:00:00 (31 archivos) y DD/MM/AAAA (2 archivos).
- Los datos incluyen datos personales: Usuario_Id, Genero y Año_de_nacimiento.